# A1 — Adaptive

**Multi-agent DAG with `share_output` state flow.** Still a static graph, but several agents now chain their results.

**Fictional task**: collaborative brief on *cultural impact of floating libraries* — research → analysis → report.

In [ ]:
# --- Load API key from the canonical env file (see memory `reference_api_keys`) ---
import os
from pathlib import Path

env_file = Path('/home/shumway/projects/meta-agents/.env')
if env_file.exists() and not os.environ.get('OPENROUTER_API_KEY'):
    for raw in env_file.read_text().splitlines():
        s = raw.strip()
        if s.startswith('OPENROUTER_API_KEY='):
            os.environ['OPENROUTER_API_KEY'] = s.split('=', 1)[1].strip().strip('"').strip("'")
            break

assert os.environ.get('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY missing'
# Default worker model for agents that do not declare their own (DAG engine consults LLM_MODEL).
os.environ.setdefault('LLM_MODEL', 'deepseek/deepseek-chat-v3.1')
print('OpenRouter key loaded. Default model:', os.environ['LLM_MODEL'])

## Load + compliance check

In [ ]:
from pathlib import Path
from awp.parser import parse_manifest, parse_agent
from awp.validator import check_compliance, AutonomyLevel

WORKFLOW_DIR = Path('/home/shumway/projects/agent-workflow-protocol/examples/workflows/02-research-pipeline')
manifest = parse_manifest(WORKFLOW_DIR / 'workflow.awp.yaml')
agents = {}
for ad in (WORKFLOW_DIR / 'agents').iterdir():
    a = ad / 'agent.awp.yaml'
    if a.exists():
        agents[ad.name] = parse_agent(a)

result = check_compliance(manifest, agents, target_level=AutonomyLevel.A1_ADAPTIVE)
assert result.level >= AutonomyLevel.A1_ADAPTIVE, f'Not A1: {result.errors}'
print(f'Compliance: {result.level_name}')
print(f'Agents: {list(agents)}')

## Run the pipeline

In [ ]:
import json
import logging
logging.basicConfig(level=logging.WARNING)

from awp.runtime import WorkflowRunner

TASK = (
    'Produce a short research brief on the (fictional) cultural impact of '
    'floating libraries in coastal cities — cover two angles and include one surprising finding.'
)
runner = WorkflowRunner(
    WORKFLOW_DIR,
    worker_model='deepseek/deepseek-chat-v3.1',
)
result = runner.run(TASK)

print(json.dumps({k: v for k, v in result.items() if not k.startswith('_')}, indent=2, default=str)[:2500])

## Assertions (E2E rubric)

In [ ]:
assert isinstance(result, dict)
agent_outputs = {
    k: v for k, v in result.items()
    if not k.startswith('_') and isinstance(v, dict) and 'confidence' in v
}
assert len(agent_outputs) >= 2, f'A1 expects >=2 agents, got: {list(agent_outputs)}'
for aid, v in agent_outputs.items():
    assert not v.get('error'), f'{aid} failed: {v["error"]}'
print(f'A1 OK — agents: {list(agent_outputs)}')